In [1]:
import sys
sys.path.append('/Users/aleksei/projects/code-of-kutulu-client')

In [2]:
!pwd

/Users/aleksei/projects/code-of-kutulu-client/notebooks


In [3]:
# import torch

In [4]:
from tqdm import tqdm
from collections import Counter
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import datetime
import pickle as pkl
import zlib
import base64
import torch

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

from src.envs.trainer import Trainer, WOOD_MAZES, BRONZE_MAZES
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS

In [6]:
def get_score(result):
    agent_id = 0
    metrics_list = result[2]
    acc = [
        [
            metrics[key][agent_id][0]
            for key in [
                'check_exp_normal', 'check_exp_coridor', 'check_exp_corner',
                'check_wan_normal', 'check_wan_coridor', 'check_wan_corner',
            ]
        ] for metrics in metrics_list[-10:]
    ]
    top_action_cnt = [
        [
            metrics[key][agent_id][2]
            for key in [
                'check_exp_normal',
                'check_wan_normal',
            ]
        ] for metrics in metrics_list[-10:]
    ]
    mean_q = [
        [
            metrics[key][agent_id][4]
            for key in [
                'check_exp_normal',
                'check_wan_normal',
            ]
        ] for metrics in metrics_list[-10:]
    ]
    score = np.mean(acc) + 0.25 * np.mean(top_action_cnt) - 0.1 * np.max(np.max(mean_q, axis=0) - np.min(mean_q, axis=0))
    return score

In [25]:
def train_evaluate(
    gamma, size, sync_target_frames, replay_start_size,
    capacity, alpha, beta,
    fc_dim, conv_dim,
    reset, reset_coef, 
    random_epsilon,
    sanity_coef, reward_for_win, reward_for_lose,
):
    env_kwargs = {
        'reward_params': {
            'sanity_coef': sanity_coef, 'reward_for_win': reward_for_win, 'reward_for_lose': reward_for_lose
        }
    }
    random_agent_info = {
        'train': False,
        'type': 'epsilon_wait',
        'action_space_n': ACTION_SPACE_N,
        'epsilon_params': {'start': random_epsilon, 'final': random_epsilon, 'decay': int(4 * 10**5)},
        'state_type': 'closest',
        'action': 'WAIT',
    }
    research_agent_info = {
        'train': True,
        'type': 'qdn_conv',
        'action_space_n': ACTION_SPACE_N,
        'state_type': 'conv',
        'batch_size': 32,
        'prioritized_replay': True,  # Enable prioritized replay
        'buffer_params': {
            'alpha': alpha,  # Control the amount of prioritization (0 = uniform, 1 = full prioritization)
            'beta': beta,   # Initial importance sampling weight (0 = no correction, 1 = full correction)
            'need_aug': True,
            'capacity': capacity,
        },
        'model_params': {
            'fc_dim': fc_dim,
            'conv_dim': conv_dim,
        },
        'sync_target_frames': sync_target_frames,
        'replay_start_size': replay_start_size,
        'epsilon_params': {
            'start': 1.0, 'final': 0.05, 'decay': int(4 * 10**5), 'reset': reset, 'reset_coef': reset_coef,
        },
        'gamma': gamma,
        'size': size,
        'lr': 1e-4,
        'optimizer': 'adamw',
        'scheduler_params': {'type': 'cosine', 'T_max': 800},
    }
    agents_info = [research_agent_info]
    for i in range(len(agents_info), 4):
        agents_info.append(dict(random_agent_info))
    assert len(agents_info) == 4
    trainer = Trainer(
        num_experiments=NUM_EXPERIMENTS, agents_info=agents_info, shuffle=True,
        league_level=LEAGUE_LEVEL, mazes=MAZES, actions=ACTIONS, log_dir='../runs', verbose=False, env_kwargs=env_kwargs,
    )
    result = trainer.train()
    return get_score(result)

In [26]:
LEAGUE_LEVEL = 2

MAZES = BRONZE_MAZES if LEAGUE_LEVEL >= 3 else WOOD_MAZES
ACTIONS = EXTENDED_KUTULU_ACTIONS if LEAGUE_LEVEL >= 3 else DEFAULT_KUTULU_ACTIONS
ACTION_SPACE_N = len(ACTIONS)
NUM_EXPERIMENTS = 10

In [27]:
import optuna
from optuna.trial import Trial

In [28]:
def objective(trial):
    params = dict(
        gamma=trial.suggest_uniform('gamma', 0.90, 0.99),
        size=trial.suggest_int('size', 1, 4),
        sync_target_frames=trial.suggest_categorical('sync_target_frames', [1000, 2000, 4000]),
        replay_start_size=trial.suggest_categorical('replay_start_size', [1000, 2000, 5000, 10000, 15000]),

        capacity=trial.suggest_categorical('capacity', [1000, 2000, 5000, 10000, 15000]),
        alpha=trial.suggest_uniform('alpha', 0.0, 1.0),
        beta=trial.suggest_uniform('beta', 0.0, 1.0),

        fc_dim=trial.suggest_categorical('fc_dim', [8, 16, 32]),
        conv_dim=trial.suggest_categorical('conv_dim', [8, 16, 32]),

        reset=trial.suggest_categorical('reset', [None, 2000, 5000, 10000, 15000]),
        reset_coef=trial.suggest_categorical('reset_coef', [1, 1.5, 2]),

        random_epsilon=trial.suggest_uniform('random_epsilon', 0.0, 1.0),

        reward_for_win=trial.suggest_categorical('reward_for_win', [None, 1, 2, 4]),
        reward_for_lose=trial.suggest_categorical('reward_for_lose', [None, -1, -2, -4]),
        sanity_coef=trial.suggest_uniform('sanity_coef', 0.05, 0.5),
    )
    return train_evaluate(**params)

In [29]:
study = optuna.create_study(direction='maximize')

[I 2025-06-21 17:34:41,743] A new study created in memory with name: no-name-c4d110f3-14b5-473e-87cf-903e2e7b92d3


In [30]:
study.optimize(objective, n_trials=10, )

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  This is separate from the ipykernel package so we can avoid doing imports until
/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/ipykernel_launcher.py:9: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  if __name__ == '__main__':
/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/ipykernel_launcher.py:10: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.


In [31]:
study.best_value

0.8125